In [3]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss
from catboost import CatBoostClassifier

def pre_df(df):
    df = df.drop(["id"], axis=1)

    df['Classic angina'] = ((df['Chest pain type'] == 4) & (df['Exercise angina'] == 1)).astype(int)
    df['High risk'] = (((df['Thallium'] == 7) | (df['Thallium'] == 6)) & (df['Number of vessels fluro'] >= 1) & (df['ST depression'] > 1)).astype(int)

    df['HR deficit'] = (df['Max HR'] + df['Age'] < 200).astype(int)

    df['Framingham risk'] = ((df['Age'] > 55) * 3 + df['Sex'] * 2 + (df['BP'] > 140) * 2 + (df['Cholesterol'] > 240) * 2 + df['FBS over 120'])
    df['Ischemic burden'] = (df['Exercise angina'] * 10 + df['ST depression'] * 5 + (3 - df['Slope of ST']) * 3)
    df['Vessel severity'] = (df['Number of vessels fluro'] * 5 + (df['Thallium'] == 6) * 3 + (df['Thallium'] == 7) * 8)

    df['Thallium chest'] = df['Thallium'] * df['Chest pain type']
    df['Age sex chest'] = df['Age'] * df['Sex'] * df['Chest pain type']
    df['Thallium vessels ST'] = df['Thallium'] * df['Number of vessels fluro'] * df['ST depression']
    df['Health'] = df['Max HR'] / 2 - df['ST depression'] * 10 - df['Exercise angina'] * 15 - df['Number of vessels fluro'] * 10
    return df

TRAIN_PATH = "../data/train.csv"
TEST_PATH = "../data/test.csv"
SUB_PATH = "../data/sample_submission.csv"

train = pre_df(pd.read_csv(TRAIN_PATH))
test = pre_df(pd.read_csv(TEST_PATH))

train["Target"] = (train["Heart Disease"] == "Presence") * 1
train = train.drop(["Heart Disease"], axis=1)

TARGET_COL = train.columns[-1]
FEATURE_COLS = list(train.columns)[:-1]
X = train[FEATURE_COLS].copy()
y = train[TARGET_COL].copy()
X_test = test[FEATURE_COLS].copy()

CAT_COLS = ["Sex", "Chest pain type", "FBS over 120", "EKG results", "Exercise angina", "Slope of ST", "Number of vessels fluro", "Thallium"] \
+ ["Classic angina", "High risk", "HR deficit", "Thallium chest", "Age sex chest", "Thallium vessels ST"]
for c in CAT_COLS:
    X[c] = X[c].astype("string")
    X_test[c] = X_test[c].astype("string")
cat_features = [X.columns.get_loc(c) for c in CAT_COLS]



In [4]:
N_SPLITS = 5
RANDOM_STATE = 42
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

test_pred = np.zeros(len(X_test))
score = 0

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        iterations=3000,
        learning_rate=0.03,
        depth=6,
        l2_leaf_reg=5.0,
        random_strength=1.0,
        bagging_temperature=1.0,
        border_count=128,
        random_seed=RANDOM_STATE + fold,
        verbose=200,
        task_type="GPU",
    )

    model.fit(
        X_train, y_train,
        cat_features=cat_features,
        eval_set=(X_valid, y_valid),
        use_best_model=True,
        early_stopping_rounds=200
    )

    test_pred += model.predict_proba(X_test)[:, 1] / N_SPLITS
    score += model.get_best_score()["validation"]["AUC"] / N_SPLITS

print(f"AUC: {score:.5f}")

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9390873	best: 0.9390873 (0)	total: 74ms	remaining: 3m 41s
200:	test: 0.9540462	best: 0.9540462 (200)	total: 11.1s	remaining: 2m 33s
400:	test: 0.9546827	best: 0.9546827 (400)	total: 21.6s	remaining: 2m 19s
600:	test: 0.9550349	best: 0.9550349 (600)	total: 32.7s	remaining: 2m 10s
800:	test: 0.9552127	best: 0.9552127 (800)	total: 45.1s	remaining: 2m 3s
1000:	test: 0.9553313	best: 0.9553313 (999)	total: 56.9s	remaining: 1m 53s
1200:	test: 0.9554334	best: 0.9554335 (1199)	total: 1m 9s	remaining: 1m 44s
1400:	test: 0.9554956	best: 0.9554956 (1400)	total: 1m 27s	remaining: 1m 39s
1600:	test: 0.9555369	best: 0.9555369 (1600)	total: 1m 47s	remaining: 1m 33s
1800:	test: 0.9555701	best: 0.9555703 (1799)	total: 2m 2s	remaining: 1m 21s
2000:	test: 0.9555917	best: 0.9555917 (1983)	total: 2m 24s	remaining: 1m 12s
2200:	test: 0.9556029	best: 0.9556029 (2200)	total: 2m 43s	remaining: 59.3s
2400:	test: 0.9556152	best: 0.9556161 (2391)	total: 3m 2s	remaining: 45.5s
2600:	test: 0.9556241	best:

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9417237	best: 0.9417237 (0)	total: 82.5ms	remaining: 4m 7s
200:	test: 0.9531612	best: 0.9531612 (200)	total: 10.7s	remaining: 2m 28s
400:	test: 0.9537844	best: 0.9537844 (400)	total: 21.8s	remaining: 2m 21s
600:	test: 0.9541322	best: 0.9541322 (600)	total: 36.1s	remaining: 2m 24s
800:	test: 0.9543160	best: 0.9543160 (800)	total: 47.4s	remaining: 2m 10s
1000:	test: 0.9544250	best: 0.9544250 (1000)	total: 1m 2s	remaining: 2m 5s
1200:	test: 0.9544997	best: 0.9544999 (1199)	total: 1m 18s	remaining: 1m 56s
1400:	test: 0.9545484	best: 0.9545486 (1399)	total: 1m 30s	remaining: 1m 42s
1600:	test: 0.9545895	best: 0.9545895 (1600)	total: 1m 43s	remaining: 1m 30s
1800:	test: 0.9546158	best: 0.9546159 (1799)	total: 1m 53s	remaining: 1m 15s
2000:	test: 0.9546430	best: 0.9546430 (2000)	total: 2m 3s	remaining: 1m 1s
2200:	test: 0.9546531	best: 0.9546541 (2180)	total: 2m 14s	remaining: 48.9s
2400:	test: 0.9546540	best: 0.9546561 (2308)	total: 2m 26s	remaining: 36.5s
2600:	test: 0.9546630	be

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9427879	best: 0.9427879 (0)	total: 193ms	remaining: 9m 38s
200:	test: 0.9539863	best: 0.9539863 (200)	total: 10.6s	remaining: 2m 28s
400:	test: 0.9545756	best: 0.9545756 (400)	total: 21.2s	remaining: 2m 17s
600:	test: 0.9549140	best: 0.9549140 (600)	total: 33.7s	remaining: 2m 14s
800:	test: 0.9550786	best: 0.9550786 (800)	total: 49.9s	remaining: 2m 17s
1000:	test: 0.9551872	best: 0.9551872 (1000)	total: 1m 7s	remaining: 2m 15s
1200:	test: 0.9552573	best: 0.9552587 (1186)	total: 1m 22s	remaining: 2m 3s
1400:	test: 0.9552945	best: 0.9552945 (1400)	total: 1m 37s	remaining: 1m 51s
1600:	test: 0.9553239	best: 0.9553239 (1598)	total: 1m 50s	remaining: 1m 36s
1800:	test: 0.9553487	best: 0.9553495 (1797)	total: 2m 11s	remaining: 1m 27s
2000:	test: 0.9553723	best: 0.9553729 (1992)	total: 2m 25s	remaining: 1m 12s
2200:	test: 0.9553839	best: 0.9553846 (2197)	total: 2m 38s	remaining: 57.5s
2400:	test: 0.9553927	best: 0.9553940 (2392)	total: 2m 52s	remaining: 43.1s
2600:	test: 0.9553977	

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9420044	best: 0.9420044 (0)	total: 121ms	remaining: 6m 4s
200:	test: 0.9534490	best: 0.9534490 (200)	total: 9.66s	remaining: 2m 14s
400:	test: 0.9540770	best: 0.9540770 (400)	total: 20s	remaining: 2m 9s
600:	test: 0.9544266	best: 0.9544266 (600)	total: 30.6s	remaining: 2m 2s
800:	test: 0.9546186	best: 0.9546186 (800)	total: 44.4s	remaining: 2m 1s
1000:	test: 0.9547344	best: 0.9547344 (1000)	total: 57.9s	remaining: 1m 55s
1200:	test: 0.9548212	best: 0.9548212 (1200)	total: 1m 12s	remaining: 1m 49s
1400:	test: 0.9548851	best: 0.9548851 (1400)	total: 1m 29s	remaining: 1m 42s
1600:	test: 0.9549201	best: 0.9549201 (1600)	total: 1m 45s	remaining: 1m 31s
1800:	test: 0.9549531	best: 0.9549531 (1800)	total: 1m 55s	remaining: 1m 16s
2000:	test: 0.9549802	best: 0.9549802 (1998)	total: 2m 5s	remaining: 1m 2s
2200:	test: 0.9549915	best: 0.9549921 (2199)	total: 2m 16s	remaining: 49.4s
2400:	test: 0.9550107	best: 0.9550107 (2400)	total: 2m 26s	remaining: 36.6s
2600:	test: 0.9550208	best: 0

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9424755	best: 0.9424755 (0)	total: 267ms	remaining: 13m 19s
200:	test: 0.9542309	best: 0.9542309 (200)	total: 11.8s	remaining: 2m 43s
400:	test: 0.9548816	best: 0.9548816 (400)	total: 22.5s	remaining: 2m 25s
600:	test: 0.9552708	best: 0.9552708 (600)	total: 33.7s	remaining: 2m 14s
800:	test: 0.9554358	best: 0.9554358 (799)	total: 45.5s	remaining: 2m 4s
1000:	test: 0.9555528	best: 0.9555528 (1000)	total: 57.4s	remaining: 1m 54s
1200:	test: 0.9556430	best: 0.9556434 (1197)	total: 1m 11s	remaining: 1m 46s
1400:	test: 0.9557043	best: 0.9557043 (1400)	total: 1m 24s	remaining: 1m 36s
1600:	test: 0.9557465	best: 0.9557467 (1598)	total: 1m 40s	remaining: 1m 27s
1800:	test: 0.9557721	best: 0.9557721 (1800)	total: 1m 54s	remaining: 1m 16s
2000:	test: 0.9557928	best: 0.9557940 (1984)	total: 2m 7s	remaining: 1m 3s
2200:	test: 0.9558103	best: 0.9558110 (2189)	total: 2m 20s	remaining: 50.9s
2400:	test: 0.9558232	best: 0.9558234 (2399)	total: 2m 34s	remaining: 38.6s
2600:	test: 0.9558346	b

In [ ]:
sub1 : 0.95546
sub2 : 0.95532

In [5]:
sub = pd.read_csv(SUB_PATH)

sub['Heart Disease'] = test_pred
sub.to_csv("../outputs/submission2.csv", index=False)